# 🎶 Music Generation - MuseGAN

In this notebook, we'll walk through the steps required to train your own MuseGAN model to generate music in the style of the Bach chorales

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-colorblind")

import os
import torch
from torch import nn
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments, TrainerCallback

from musegan_utils import notes_to_midi, draw_score

# Check GPU - the Hugging Face Trainer picks the best available device
# automatically, so we only report what it will find
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
BATCH_SIZE = 64

N_BARS = 2
N_STEPS_PER_BAR = 16
MAX_PITCH = 83
N_PITCHES = MAX_PITCH + 1
Z_DIM = 32


CRITIC_STEPS = 5
GP_WEIGHT = 10
CRITIC_LEARNING_RATE = 0.001
GENERATOR_LEARNING_RATE = 0.001
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.9
EPOCHS = 6000
LOAD_MODEL = False

## 1. Prepare the Data

In [ ]:
# Load the data
file = os.path.join("../../../data/bach-chorales/Jsb16thSeparated.npz")
with np.load(file, encoding="bytes", allow_pickle=True) as f:
    data = f["train"]

In [ ]:
N_SONGS = len(data)
print(f"{N_SONGS} chorales in the dataset")
chorale = data[0]
N_BEATS, N_TRACKS = chorale.shape
print(f"{N_BEATS, N_TRACKS} shape of chorale 0")
print("\nChorale 0")
print(chorale[:8])

In [ ]:
two_bars = np.array([x[: (N_STEPS_PER_BAR * N_BARS)] for x in data])
two_bars = np.array(np.nan_to_num(two_bars, nan=MAX_PITCH), dtype=int)
two_bars = two_bars.reshape([N_SONGS, N_BARS, N_STEPS_PER_BAR, N_TRACKS])
print(f"Two bars shape {two_bars.shape}")

In [ ]:
data_binary = np.eye(N_PITCHES)[two_bars]  # (songs, bars, steps, tracks, pitches)
data_binary[data_binary == 0] = -1
# PyTorch's Conv3d wants channels first: the 4 tracks are the channels and
# (bars, steps, pitches) the 3 "spatial" dimensions
data_binary = data_binary.transpose([0, 3, 1, 2, 4])
print(f"Data binary shape {data_binary.shape}")


# The Trainer passes each batch to the model as a dict, so every example is a dict
class ChoraleDataset(Dataset):
    def __init__(self, scores):
        self.scores = torch.tensor(scores, dtype=torch.float32)

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):
        return {"real_images": self.scores[idx]}


train_dataset = ChoraleDataset(data_binary)


def to_score(music):
    # (batch, tracks, bars, steps, pitches) tensor -> the (batch, bars, steps, pitches, tracks)
    # NumPy layout that notes_to_midi / draw_score from musegan_utils expect
    return music.permute(0, 2, 3, 4, 1).detach().cpu().numpy()

## 2. Build the GAN <a name="build"></a>

In [ ]:
# Some helper functions


def init_weights(layer):
    # Same as Keras' RandomNormal(mean=0.0, stddev=0.02) kernel initializer (biases start at 0)
    nn.init.normal_(layer.weight, mean=0.0, std=0.02)
    nn.init.zeros_(layer.bias)
    return layer


def conv(in_f, f, k, s, p):
    # Keras' LeakyReLU has a default negative slope of 0.3 (PyTorch: 0.01)
    return nn.Sequential(
        init_weights(nn.Conv3d(in_f, f, kernel_size=k, stride=s, padding=p)),
        nn.LeakyReLU(0.3),
    )


def conv_t(in_f, f, k, s, a, bn):
    # Every transposed conv in this notebook has kernel_size == stride (or is "valid"),
    # so no padding is needed for the output to be exactly input * stride
    layers = [init_weights(nn.ConvTranspose2d(in_f, f, kernel_size=k, stride=s))]
    if bn:
        layers.append(nn.BatchNorm2d(f, momentum=0.1))  # Keras momentum 0.9 == PyTorch 0.1
    layers.append(a)
    return nn.Sequential(*layers)

In [ ]:
class TemporalNetwork(nn.Module):
    # z of shape (batch, Z_DIM) -> one z vector per bar: (batch, N_BARS, Z_DIM)
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Unflatten(1, (Z_DIM, 1, 1)),  # (B, Z_DIM, 1, 1)
            conv_t(Z_DIM, 1024, k=(2, 1), s=(1, 1), a=nn.ReLU(), bn=True),
            conv_t(1024, Z_DIM, k=(N_BARS - 1, 1), s=(1, 1), a=nn.ReLU(), bn=True),
        )  # -> (B, Z_DIM, N_BARS, 1)

    def forward(self, x):
        return self.net(x).squeeze(-1).transpose(1, 2)  # (B, N_BARS, Z_DIM)


print(TemporalNetwork())
print(f"Total params: {sum(p.numel() for p in TemporalNetwork().parameters()):,}")

In [ ]:
class BarGenerator(nn.Module):
    # (batch, Z_DIM * 4) -> one bar of one track: (batch, 1 track, 1 bar, N_STEPS_PER_BAR, N_PITCHES)
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(Z_DIM * 4, 1024),
            nn.BatchNorm1d(1024, momentum=0.1),
            nn.ReLU(),
            nn.Unflatten(1, (512, 2, 1)),  # (B, 512, 2, 1)
            conv_t(512, 512, k=(2, 1), s=(2, 1), a=nn.ReLU(), bn=True),  # (B, 512, 4, 1)
            conv_t(512, 256, k=(2, 1), s=(2, 1), a=nn.ReLU(), bn=True),  # (B, 256, 8, 1)
            conv_t(256, 256, k=(2, 1), s=(2, 1), a=nn.ReLU(), bn=True),  # (B, 256, 16, 1)
            conv_t(256, 256, k=(1, 7), s=(1, 7), a=nn.ReLU(), bn=True),  # (B, 256, 16, 7)
            conv_t(256, 1, k=(1, 12), s=(1, 12), a=nn.Tanh(), bn=False),  # (B, 1, 16, 84)
        )

    def forward(self, x):
        return self.net(x).unsqueeze(2)  # add the bar dimension


print(BarGenerator())
print(f"Total params: {sum(p.numel() for p in BarGenerator().parameters()):,}")

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        # CHORDS -> TEMPORAL NETWORK
        self.chords_tempNetwork = TemporalNetwork()
        # MELODY -> TEMPORAL NETWORK (one per track)
        self.melody_tempNetwork = nn.ModuleList(
            [TemporalNetwork() for _ in range(N_TRACKS)]
        )
        # CREATE BAR GENERATOR FOR EACH TRACK
        self.barGen = nn.ModuleList([BarGenerator() for _ in range(N_TRACKS)])

    def forward(self, chords_input, style_input, melody_input, groove_input):
        chords_over_time = self.chords_tempNetwork(chords_input)  # [n_bars, z_dim]
        melody_over_time = [
            self.melody_tempNetwork[track](melody_input[:, track, :])
            for track in range(N_TRACKS)
        ]  # list of n_tracks [n_bars, z_dim] tensors

        # CREATE OUTPUT FOR EVERY TRACK AND BAR
        bars_output = []
        for bar in range(N_BARS):
            c = chords_over_time[:, bar, :]  # [z_dim]
            s = style_input  # [z_dim]
            track_output = []
            for track in range(N_TRACKS):
                m = melody_over_time[track][:, bar, :]  # [z_dim]
                g = groove_input[:, track, :]  # [z_dim]
                z_input = torch.cat([c, s, m, g], dim=1)
                track_output.append(self.barGen[track](z_input))
            bars_output.append(torch.cat(track_output, dim=1))  # stack tracks (channels)

        # (batch, N_TRACKS, N_BARS, N_STEPS_PER_BAR, N_PITCHES)
        return torch.cat(bars_output, dim=2)


generator = Generator()
# print(generator)

In [ ]:
class Critic(nn.Module):
    # Input: (batch, N_TRACKS, N_BARS, N_STEPS_PER_BAR, N_PITCHES), channels first
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            conv(N_TRACKS, 128, k=(2, 1, 1), s=(1, 1, 1), p=0),  # (B, 128, 1, 16, 84)
            conv(128, 128, k=(N_BARS - 1, 1, 1), s=(1, 1, 1), p=0),
            conv(128, 128, k=(1, 1, 12), s=(1, 1, 12), p=0),  # (B, 128, 1, 16, 7)
            conv(128, 128, k=(1, 1, 7), s=(1, 1, 7), p=0),  # (B, 128, 1, 16, 1)
            conv(128, 128, k=(1, 2, 1), s=(1, 2, 1), p=0),  # (B, 128, 1, 8, 1)
            conv(128, 128, k=(1, 2, 1), s=(1, 2, 1), p=0),  # (B, 128, 1, 4, 1)
            conv(128, 256, k=(1, 4, 1), s=(1, 2, 1), p=(0, 1, 0)),  # (B, 256, 1, 2, 1)
            conv(256, 512, k=(1, 3, 1), s=(1, 2, 1), p=(0, 1, 0)),  # (B, 512, 1, 1, 1)
            nn.Flatten(),
            init_weights(nn.Linear(512, 1024)),
            nn.LeakyReLU(0.3),
            init_weights(nn.Linear(1024, 1)),  # no activation: WGAN critic outputs a score
        )

    def forward(self, x):
        return self.net(x)


critic = Critic()
# print(critic)

In [ ]:
class MuseGAN(nn.Module):
    # A plain nn.Module bundling the two networks and the losses; the custom
    # MuseGANTrainer below decides how they are optimized
    def __init__(self, critic, generator, latent_dim, critic_steps, gp_weight):
        super().__init__()
        self.critic = critic
        self.generator = generator
        self.latent_dim = latent_dim
        self.critic_steps = critic_steps
        self.gp_weight = gp_weight

    def random_latent_vectors(self, batch_size):
        device = next(self.parameters()).device
        return [
            torch.randn(batch_size, self.latent_dim, device=device),  # chords
            torch.randn(batch_size, self.latent_dim, device=device),  # style
            torch.randn(batch_size, N_TRACKS, self.latent_dim, device=device),  # melody
            torch.randn(batch_size, N_TRACKS, self.latent_dim, device=device),  # groove
        ]

    def gradient_penalty(self, batch_size, real_images, fake_images):
        # Standard WGAN-GP: alpha ~ U(0, 1) and the gradient norm over all
        # (track, bar, step, pitch) values of each sample
        alpha = torch.rand(batch_size, 1, 1, 1, 1, device=real_images.device)
        diff = fake_images - real_images
        interpolated = (real_images + alpha * diff).requires_grad_(True)

        pred = self.critic(interpolated)

        # create_graph=True so we can backpropagate through the gradient itself
        grads = torch.autograd.grad(pred.sum(), interpolated, create_graph=True)[0]
        norm = grads.flatten(1).norm(2, dim=1)
        gp = ((norm - 1.0) ** 2).mean()
        return gp

    def critic_loss(self, real_images):
        batch_size = real_images.shape[0]
        with torch.no_grad():  # the generator is not updated in the critic step
            fake_images = self.generator(*self.random_latent_vectors(batch_size))
        fake_predictions = self.critic(fake_images)
        real_predictions = self.critic(real_images)

        c_wass_loss = fake_predictions.mean() - real_predictions.mean()
        c_gp = self.gradient_penalty(batch_size, real_images, fake_images)
        c_loss = c_wass_loss + c_gp * self.gp_weight
        return c_loss, c_wass_loss, c_gp

    def generator_loss(self, batch_size):
        fake_images = self.generator(*self.random_latent_vectors(batch_size))
        fake_predictions = self.critic(fake_images)
        return -fake_predictions.mean()

    @torch.no_grad()
    def generate_piano_roll(self, num_scores):
        self.generator.eval()  # BatchNorm uses its running statistics
        generated_music = self.generator(*self.random_latent_vectors(num_scores))
        self.generator.train()
        return to_score(generated_music)

In [ ]:
# Create a MuseGAN
musegan = MuseGAN(
    critic=critic,
    generator=generator,
    latent_dim=Z_DIM,
    critic_steps=CRITIC_STEPS,
    gp_weight=GP_WEIGHT,
)

In [ ]:
if LOAD_MODEL:
    # The Trainer saves checkpoints as ./checkpoint/checkpoint-<step>/model.safetensors
    from safetensors.torch import load_file
    from transformers.trainer_utils import get_last_checkpoint

    musegan.load_state_dict(
        load_file(os.path.join(get_last_checkpoint("./checkpoint"), "model.safetensors"))
    )

## 3. Train the MuseGAN <a name="train"></a>

In [ ]:
# The standard Trainer does one forward/backward pass of a single loss and one
# optimizer step per batch. A WGAN-GP needs two optimizers and CRITIC_STEPS critic
# updates per generator update, so we subclass Trainer and override:
#  - create_optimizer: build both Adam optimizers
#  - training_step:    run the critic steps ourselves and return the generator loss,
#                      which the Trainer then steps with self.optimizer (= g_optimizer)
class MuseGANTrainer(Trainer):
    def create_optimizer(self, model=None):
        self.c_optimizer = torch.optim.Adam(
            self.model.critic.parameters(),
            lr=CRITIC_LEARNING_RATE,
            betas=(ADAM_BETA_1, ADAM_BETA_2),
        )
        self.optimizer = torch.optim.Adam(
            self.model.generator.parameters(),
            lr=GENERATOR_LEARNING_RATE,
            betas=(ADAM_BETA_1, ADAM_BETA_2),
        )
        self.critic_logs = []
        return self.optimizer

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        real_images = inputs["real_images"]

        # 1. Train the critic CRITIC_STEPS times on this batch
        for i in range(model.critic_steps):
            c_loss, c_wass_loss, c_gp = model.critic_loss(real_images)
            self.c_optimizer.zero_grad()
            c_loss.backward()
            self.c_optimizer.step()
        self.critic_logs.append([c_loss.item(), c_wass_loss.item(), c_gp.item()])

        # 2. Generator loss: the Trainer calls g_optimizer.step() and zero_grad() after this
        g_loss = model.generator_loss(len(real_images))
        g_loss.backward()
        return g_loss.detach()

    def log(self, logs, *args, **kwargs):
        # "loss" is the generator loss; add the averaged critic losses to the epoch logs
        if "loss" in logs and self.critic_logs:
            means = np.mean(self.critic_logs, axis=0)
            logs.update(c_loss=means[0], c_wass_loss=means[1], c_gp=means[2])
            self.critic_logs = []
        super().log(logs, *args, **kwargs)


os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"  # where report_to="tensorboard" writes
training_args = TrainingArguments(
    output_dir="./checkpoint",
    lr_scheduler_type="constant",  # Keras keeps the learning rate fixed
    max_grad_norm=0.0,  # Keras doesn't clip gradients (the Trainer clips at 1.0 by default)
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    remove_unused_columns=False,  # MuseGAN has no forward(), so keep every batch key
    logging_strategy="epoch",
    save_strategy="epoch",  # save a checkpoint after every epoch...
    save_total_limit=1,  # ...keeping only the latest one
    report_to="tensorboard",
)

In [ ]:
class MusicGenerator(TrainerCallback):
    def __init__(self, num_scores):
        self.num_scores = num_scores

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch) - 1  # state.epoch counts finished epochs
        if epoch % 1 == 0:
            generated_music = model.generate_piano_roll(self.num_scores)
            notes_to_midi(
                generated_music,
                N_BARS,
                N_TRACKS,
                N_STEPS_PER_BAR,
                filename="output_" + str(epoch).zfill(4),
            )


music_generator_callback = MusicGenerator(1)

trainer = MuseGANTrainer(
    model=musegan,
    args=training_args,
    train_dataset=train_dataset,
    callbacks=[music_generator_callback],
)

In [ ]:
history = trainer.train()

# Generate new scores

In [ ]:
num_scores = 1
chords_random_latent_vectors = np.random.normal(size=(num_scores, Z_DIM))
style_random_latent_vectors = np.random.normal(size=(num_scores, Z_DIM))
melody_random_latent_vectors = np.random.normal(
    size=(num_scores, N_TRACKS, Z_DIM)
)
groove_random_latent_vectors = np.random.normal(
    size=(num_scores, N_TRACKS, Z_DIM)
)
random_latent_vectors = [
    chords_random_latent_vectors,
    style_random_latent_vectors,
    melody_random_latent_vectors,
    groove_random_latent_vectors,
]


@torch.no_grad()
def generate(random_latent_vectors):
    # NumPy latent vectors -> float32 tensors on the model's device, then run the
    # generator in eval mode and return the (batch, bars, steps, pitches, tracks) score
    generator.eval()
    device = next(generator.parameters()).device
    inputs = [torch.tensor(z, dtype=torch.float32, device=device) for z in random_latent_vectors]
    return to_score(generator(*inputs))


generated_music = generate(random_latent_vectors)

draw_score(generated_music, 0)
notes_to_midi(
    generated_music, N_BARS, N_TRACKS, N_STEPS_PER_BAR, filename="output_midi"
)

## Changing Chord Noise

In [ ]:
chords_random_latent_vectors_2 = np.random.normal(size=(num_scores, Z_DIM))
random_latent_vectors_2 = [
    chords_random_latent_vectors_2,
    style_random_latent_vectors,
    melody_random_latent_vectors,
    groove_random_latent_vectors,
]
generated_music_2 = generate(random_latent_vectors_2)
draw_score(generated_music_2, 0)
notes_to_midi(
    generated_music_2,
    N_BARS,
    N_TRACKS,
    N_STEPS_PER_BAR,
    filename="output_midi_chords_changed",
)

# Changing Style Noise

In [ ]:
style_random_latent_vectors_2 = np.random.normal(size=(num_scores, Z_DIM))
random_latent_vectors_3 = [
    chords_random_latent_vectors,
    style_random_latent_vectors_2,
    melody_random_latent_vectors,
    groove_random_latent_vectors,
]
generated_music_3 = generate(random_latent_vectors_3)
draw_score(generated_music_3, 0)
notes_to_midi(
    generated_music_3,
    N_BARS,
    N_TRACKS,
    N_STEPS_PER_BAR,
    filename="output_midi_style_changed",
)

## Changing Melody Noise

In [ ]:
melody_random_latent_vectors_2 = np.copy(melody_random_latent_vectors)
melody_random_latent_vectors_2[:, 0, :] = np.random.normal(
    size=(num_scores, Z_DIM)
)

random_latent_vectors_4 = [
    chords_random_latent_vectors,
    style_random_latent_vectors,
    melody_random_latent_vectors_2,
    groove_random_latent_vectors,
]
generated_music_4 = generate(random_latent_vectors_4)
draw_score(generated_music_4, 0)
notes_to_midi(
    generated_music_4,
    N_BARS,
    N_TRACKS,
    N_STEPS_PER_BAR,
    filename="output_midi_melody_changed",
)

## Changing groove noise

In [ ]:
groove_random_latent_vectors_2 = np.copy(groove_random_latent_vectors)
groove_random_latent_vectors_2[:, -1, :] = np.random.normal(
    size=(num_scores, Z_DIM)
)

random_latent_vectors_5 = [
    chords_random_latent_vectors,
    style_random_latent_vectors,
    melody_random_latent_vectors,
    groove_random_latent_vectors_2,
]
generated_music_5 = generate(random_latent_vectors_5)
draw_score(generated_music_5, 0)
notes_to_midi(
    generated_music_5,
    N_BARS,
    N_TRACKS,
    N_STEPS_PER_BAR,
    filename="output_midi_groove_changed",
)